# 🛡️ EPI Finder - Pipeline de Inferência Operacional e Relatórios de Conformidade (Fase 6)

Bem-vindo à **Fase 6** do projeto **EPI Finder**!

Após prepararmos os dados, realizarmos a Análise Exploratória (EDA), treinarmos o modelo YOLOv8 e auditarmos suas métricas na Fase 5, chegamos à **etapa de entrega de valor operacional**.

O objetivo central da Fase 6 é transformar a rede neural em uma **ferramenta de monitoramento operacional de Segurança e Saúde no Trabalho (SST)**. Esta aplicação recebe fotos ou transmissões de vídeo de canteiros industriais, identifica o uso de capacete de proteção, renderiza **alertas visuais instantâneos** e gera **relatórios tabulares de auditoria** consolidados com **Pandas** e **NumPy**.

---

### 🎯 Tópicos Abordados neste Caderno:
1. **Configuração do Pipeline:** Carregamento de módulos operacionais (`src.utils` e `src.inference`).
2. **Anatomia Visual do Alerta (OpenCV):** Bounding boxes dinâmicas em **Vermelho** (`ALERTA: SEM CAPACETE`) e **Verde** (`Capacete`).
3. **Telemetria de Segurança no Frame:** Sobreposição de painel superior semi-transparente (*alpha blending*) com indicadores em tempo real.
4. **Recorte de Evidências com NumPy (ROI Slicing):** Extração matricial das infrações para auditoria humana de SST.
5. **Processamento em Lote e Registro de Eventos:** Execução do motor de inferência sobre múltiplas imagens.
6. **Auditoria Estatística com Pandas:** Análise aprofundada do relatório tabular `compliance_report.csv` com gráficos de conformidade, distribuição de confiança e volumetria de infrações.
7. **Consolidação do Projeto e Próximos Passos (Fase 7):** Boas práticas para deploy contínuo e extensões para CFTV.

## 1. Configuração do Ambiente e Importação de Bibliotecas

Configuramos o caminho raiz do projeto no `sys.path`, definimos a paleta visual elegante para os gráficos com **Matplotlib/Seaborn** e importamos os módulos operacionais reutilizáveis desenvolvidos em `src/`.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Ultralytics YOLO
import ultralytics
from ultralytics import YOLO

# Garante que a raiz do projeto esteja no sys.path e seja o diretório de trabalho atual
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Módulos customizados do EPI Finder
from src.utils import (
    ALERT_CLASSES,
    CLASS_COLORS_BGR,
    DEFAULT_CLASSES,
    draw_bounding_boxes,
    draw_telemetry_banner,
    extract_roi,
    bgr_to_rgb
)
from src.inference import (
    ComplianceAuditor,
    run_inference
)

# Configurações visuais para gráficos de auditoria
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print(f"📁 Diretório raiz do projeto : {PROJECT_ROOT}")
print(f"🐍 Python                    : {sys.version.split()[0]}")
print(f"📊 Pandas versão             : {pd.__version__}")
print(f"🚀 Ultralytics versão        : {ultralytics.__version__}")
print("✅ Módulos 'src.utils' e 'src.inference' carregados com sucesso!")

## 2. Carregamento do Modelo Treinado e Metadados

Verificamos a existência do checkpoint treinado em `models/best.pt` e seus metadados de auditoria em `models/metadata.json`.

In [ ]:
weights_path = PROJECT_ROOT / "models" / "best.pt"
metadata_path = PROJECT_ROOT / "models" / "metadata.json"

if not weights_path.exists():
    print(f"⚠️  Pesos '{weights_path.name}' não encontrados. Usando fallback 'yolov8n.pt'...")
    weights_path = PROJECT_ROOT / "yolov8n.pt"

print(f"📦 Checkpoint selecionado: {weights_path}")
model = YOLO(str(weights_path))

if metadata_path.exists():
    with open(metadata_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print(f"📋 Modelo Treinado : {meta.get('model_name')}")
    print(f"📅 Data do Treino  : {meta.get('training_date', 'N/A')[:19]}")
    print(f"🏷️  Classes          : {meta.get('classes')}")

## 3. Anatomia Visual da Inferência Operacional (OpenCV)

Vamos realizar a inferência em uma imagem real do conjunto de teste (`data/dataset/test/images/`) e inspecionar a geração dos alertas visuais:
- Bounding box **Vermelha** para `0: head` com a legenda `"ALERTA: SEM CAPACETE"`.
- Bounding box **Verde** para `1: helmet` com a legenda `"Capacete"`.
- Banner superior de telemetria semi-transparente contendo o identificador do posto, data/hora e percentual de conformidade.

In [ ]:
# Selecionar uma imagem de teste representativa
test_images = sorted(list((PROJECT_ROOT / "data" / "dataset" / "test" / "images").glob("*.jpg")))
sample_image_path = test_images[0] if test_images else None

print(f"🖼️  Imagem selecionada para demonstração: {sample_image_path.name}")

# Leitura com OpenCV (formato BGR)
img_bgr = cv2.imread(str(sample_image_path))
h, w, c = img_bgr.shape

# Execução da predição com YOLO
# Nota: calibramos conf=0.001 se estivermos auditando modelo de smoke test
results = model.predict(
    source=img_bgr,
    conf=0.001,
    iou=0.45,
    imgsz=640,
    verbose=False
)

boxes_xyxy = []
class_ids = []
confidences = []

for r in results:
    if r.boxes is not None and len(r.boxes) > 0:
        b_arr = r.boxes.xyxy.cpu().numpy()
        c_arr = r.boxes.cls.cpu().numpy().astype(int)
        conf_arr = r.boxes.conf.cpu().numpy()
        for b, c_id, cf in zip(b_arr, c_arr, conf_arr):
            boxes_xyxy.append((int(b[0]), int(b[1]), int(b[2]), int(b[3])))
            class_ids.append(int(c_id))
            confidences.append(float(cf))

total = len(class_ids)
helmets = sum(1 for c in class_ids if c == 1)
violations = sum(1 for c in class_ids if c == 0)
comp_rate = (helmets / total * 100.0) if total > 0 else 100.0

print(f"👥 Detecções no quadro : {total}")
print(f"✅ Com Capacete        : {helmets}")
print(f"❌ Sem Capacete (EPI)  : {violations}")
print(f"📈 Taxa de Conformidade: {comp_rate:.1f}%")

# Renderização com OpenCV
annotated = img_bgr.copy()
if boxes_xyxy:
    annotated = draw_bounding_boxes(
        image=annotated,
        boxes_xyxy=boxes_xyxy,
        class_ids=class_ids,
        confidences=confidences,
        class_names=ALERT_CLASSES,
        thickness=2
    )

# Sobreposição do banner de telemetria
annotated = draw_telemetry_banner(
    image=annotated,
    total_persons=total,
    conformant_count=helmets,
    violation_count=violations,
    compliance_rate=comp_rate,
    camera_id="Portaria Canteiro 01",
    timestamp_str="2026-08-30 14:00:00"
)

# Exibição comparativa Lado a Lado com Matplotlib
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(bgr_to_rgb(img_bgr))
axes[0].set_title("Imagem Original (Câmera)", fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(bgr_to_rgb(annotated))
axes[1].set_title("Feed com Telemetria e Alertas Operacionais (EPI Finder)", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Recorte Matricial de Evidências com NumPy (ROI Slicing)

Na gestão de **Segurança do Trabalho**, apenas alertar na tela não é suficiente: a equipe de auditoria precisa de **evidências visuais** para orientar e conscientizar os trabalhadores infratores.

A função `extract_roi` do módulo `src.utils` utiliza **fatiamento direto de matrizes NumPy** (`img[y1:y2, x1:x2]`) para recortar instantaneamente as cabeças desprotegidas detectadas.

In [ ]:
# Filtrar as caixas que representam infrações (class_id == 0: head)
violation_boxes = [box for box, c_id in zip(boxes_xyxy, class_ids) if c_id == 0]

print(f"🚨 Total de infrações identificadas para recorte: {len(violation_boxes)}")

if violation_boxes:
    num_crops = min(6, len(violation_boxes))
    fig, axes = plt.subplots(1, num_crops, figsize=(3 * num_crops, 3))
    if num_crops == 1:
        axes = [axes]

    for i in range(num_crops):
        box = violation_boxes[i]
        crop = extract_roi(img_bgr, box)
        if crop.size > 0:
            axes[i].imshow(bgr_to_rgb(crop))
            axes[i].set_title(f"Infração #{i+1}\n{box[2]-box[0]}x{box[3]-box[1]} px", fontsize=10, color='red')
        axes[i].axis('off')

    plt.suptitle("Galeria de Evidências de Não-Conformidade (Recortes NumPy)", fontsize=13, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma infração presente para recorte nesta amostra.")

## 5. Execução do Pipeline Operacional em Lote

Agora acionamos o motor de inferência [`run_inference`](../src/inference.py) para processar o lote completo de teste e salvar:
1. As imagens anotadas em `runs/inference/`.
2. Os recortes de infrações em `runs/inference/violations/`.
3. O relatório detalhado de conformidade em `runs/inference/compliance_report.csv`.
4. O sumário executivo de auditoria em `runs/inference/summary.json`.

In [ ]:
output_dir = PROJECT_ROOT / "runs" / "inference"
report_csv = output_dir / "compliance_report.csv"
summary_json = output_dir / "summary.json"

# Executar inferência em lote
auditor = run_inference(
    source=PROJECT_ROOT / "data" / "dataset" / "test" / "images",
    weights=weights_path,
    conf_threshold=0.001,
    iou_threshold=0.45,
    img_size=640,
    output_dir=output_dir,
    report_csv=report_csv,
    summary_json=summary_json,
    camera_id="Câmera 01 - Portaria Central",
    save_crops=True,
    save_media=True,
    alert_labels=True,
    draw_banner=True,
    verbose=True
)

## 6. Auditoria Estatística e Análise de Conformidade com Pandas

Com o relatório consolidado gerado pelo pipeline, utilizamos o **Pandas** para realizar uma auditoria detalhada dos indicadores de segurança.

In [ ]:
# Carregamento do relatório CSV gerado pelo auditor
df_report = pd.read_csv(report_csv)

print(f"📊 Total de registros de detecção gerados: {len(df_report):,}")
print("\nVisualização das primeiras 5 linhas do relatório:")
display(df_report.head())

# Tipos de dados e conferência de nulos
print("\nInformações estruturais do DataFrame:")
df_report.info()

### 6.1. Indicadores Principais de SST (KPIs de Segurança)

In [ ]:
total_detections = len(df_report)
helmets_count = int((df_report["class_id"] == 1).sum())
violations_count = int((df_report["class_id"] == 0).sum())
compliance_rate = (helmets_count / total_detections * 100.0) if total_detections > 0 else 100.0

kpis = pd.DataFrame({
    "Indicador": [
        "Total de Pessoas Detectadas",
        "Pessoas em Conformidade (Capacete)",
        "Infrações Detectadas (Sem Capacete)",
        "Taxa Global de Conformidade (%)",
        "Confiança Média do Detector"
    ],
    "Valor": [
        f"{total_detections:,}",
        f"{helmets_count:,}",
        f"{violations_count:,}",
        f"{compliance_rate:.2f}%",
        f"{df_report['confidence'].mean():.4f}"
    ]
})

display(kpis)

### 6.2. Visualizações Analíticas de Conformidade

Construímos visualizações para auditar a conformidade de EPI e a segurança operacional:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Gráfico de Rosca: Taxa de Conformidade
labels = ['Conforme (Capacete)', 'Infração (Sem Capacete)']
sizes = [helmets_count, violations_count]
colors = ['#2ecc71', '#e74c3c']

wedges, texts, autotexts = axes[0].pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2)
)
plt.setp(autotexts, size=12, weight="bold")
axes[0].set_title("Índice Geral de Conformidade NR-6", fontsize=13, fontweight='bold')

# 2. Histograma de Distribuição de Confiança por Classe
sns.histplot(
    data=df_report,
    x='confidence',
    hue='class_name',
    palette={'helmet': '#2ecc71', 'head': '#e74c3c'},
    bins=25,
    kde=True,
    ax=axes[1]
)
axes[1].set_title("Distribuição de Confiança das Detecções", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Score de Confiança")
axes[1].set_ylabel("Frequência")

# 3. Relação entre Área da Caixa (Pixels) e Confiança
sns.scatterplot(
    data=df_report,
    x='box_area',
    y='confidence',
    hue='class_name',
    palette={'helmet': '#2ecc71', 'head': '#e74c3c'},
    alpha=0.6,
    ax=axes[2]
)
axes[2].set_title("Área da Caixa vs. Confiança", fontsize=13, fontweight='bold')
axes[2].set_xlabel("Área da Bounding Box (px²)")
axes[2].set_ylabel("Confiança")

plt.tight_layout()
plt.show()

## 7. Conclusão da Fase 6 e Visão Geral do Pipeline

### 🏆 Resultados Consolidados:
1. **Módulo Operacional Criado (`src/inference.py`):** Suporte completo para inferência em fotos, diretórios, vídeos e webcams.
2. **Alertas em Tempo Real:** Destaque visual imediato em vermelho para infrações (`head`) e verde para conformidade (`helmet`), com telemetria no topo do frame.
3. **Recorte de Evidências:** Salvamento automático de fotos recortadas de pessoas sem capacete para comprovação e treinamento interno da CIPA / SST.
4. **Auditoria Tabular:** Geração estruturada de `compliance_report.csv` permitindo integração com bancos de dados, dashboards em Power BI ou painéis corporativos.

---

### 🚀 Possíveis Extensões (Fase 7 - Opcional):
- **Rastreamento de Objetos (Object Tracking):** Integrar algoritmos como ByteTrack ou BoT-SORT (`model.track(source=...)`) para rastrear o mesmo trabalhador entre frames e evitar contagens duplicadas no relatório diário.
- **Dashboard Web Interativo:** Criar uma interface web com **Streamlit** onde o usuário pode fazer upload de um vídeo e acompanhar os gráficos interativos em tempo real.